# EcoSort AI - Local Run

This notebook runs the original saved 12-class model on this Mac without retraining or changing the classifier. Use the EcoSort Local kernel. The complete 58-cell training notebook and unchanged Colab export are separate files in this folder.

## 1. Load the saved model and original calibration

Load the exact model weights, category order, temperature and review threshold from the downloaded evidence archive. Print available GPU devices. No cloud connection is needed for predictions.


In [ ]:
from pathlib import Path
import json
import os
import time
import numpy as np
import tensorflow as tf
import gradio as gr
from PIL import Image, ImageOps
from scipy.special import softmax

PROJECT = Path.cwd()
RESULTS = PROJECT / "saved_results"
assert RESULTS.is_dir(), "Open this notebook from the EcoSort-Local project folder."
settings = json.loads((RESULTS / "app_settings.json").read_text())
LABELS_V3 = json.loads((RESULTS / "class_names.json").read_text())
assert settings["classes"] == LABELS_V3 and len(LABELS_V3) == 12
temperature3 = float(settings["temperature"])
review_threshold3 = float(settings["review_threshold"])
weighted_model3 = tf.keras.models.load_model(RESULTS / "ecosort_weighted_12class.keras", compile=False)
assert weighted_model3.output_shape[-1] == 12
print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))
print("Loaded unchanged saved 12-class model; no retraining.")


## 2. Original prediction function

The following definition is copied verbatim from Colab cell 52. It corrects EXIF orientation, converts to RGB, resizes to 224 by 224, applies the original model and temperature scaling, and requests manual review for low-confidence or closely competing predictions.


In [ ]:
DISPLAY3 = ["Battery", "Cardboard", "Ceramics", "Electronic waste", "Food organics", "Glass", "Metal", "Paper", "Plastic", "Textile waste", "Other waste", "Vegetation"]
def classify_balanced_waste(image):
    if image is None:
        raise gr.Error("Upload an image or take a photo first.")
    rgb = ImageOps.exif_transpose(image).convert("RGB")
    pixels = tf.image.resize(np.asarray(rgb, dtype=np.float32), (224, 224))
    raw_scores = weighted_model3(np.expand_dims(pixels.numpy(), 0), training=False).numpy()[0]
    scores = softmax(np.log(np.clip(raw_scores, 1e-8, 1)) / temperature3)
    if scores.shape != (12,) or not np.isfinite(scores).all():
        raise gr.Error("Invalid model output; please try another image.")
    order = np.argsort(scores)
    winner = int(order[-1])
    uncertain = float(scores[winner]) < review_threshold3 or float(scores[order[-1]] - scores[order[-2]]) < 0.10
    decision = "Uncertain / manual review" if uncertain else DISPLAY3[winner]
    message = f"{decision} | Top suggestion: {DISPLAY3[winner]} | score: {scores[winner]:.1%}"
    return dict(zip(DISPLAY3, map(float, scores))), message


## 3. Run and record a local smoke test

Download one public TrashNet cardboard sample on first run, verify the 12 scores sum to one, check missing-image validation, and time five warm predictions. This confirms local operation, not model accuracy on a new dataset. No webcam permissions are requested by this test.


In [ ]:
import urllib.request
import ssl
import certifi
import zipfile
import hashlib

SAMPLES = PROJECT / "samples"
SAMPLES.mkdir(exist_ok=True)
SAMPLE = SAMPLES / "cardboard1.jpg"
if not SAMPLE.exists():
    context = ssl.create_default_context(cafile=certifi.where())
    url = "https://raw.githubusercontent.com/garythung/trashnet/master/data/dataset-resized.zip"
    with urllib.request.urlopen(url, context=context, timeout=120) as response:
        archive_data = response.read()
    import io
    with zipfile.ZipFile(io.BytesIO(archive_data)) as archive:
        SAMPLE.write_bytes(archive.read("dataset-resized/cardboard/cardboard1.jpg"))

with Image.open(SAMPLE) as sample:
    scores_check3, message_check3 = classify_balanced_waste(sample)
assert len(scores_check3) == 12 and np.isclose(sum(scores_check3.values()), 1.0)
assert all(np.isfinite(list(scores_check3.values())))
try:
    classify_balanced_waste(None)
except gr.Error:
    pass
else:
    raise AssertionError("Missing-image validation must reject empty input.")
durations = []
with Image.open(SAMPLE) as sample:
    for _ in range(5):
        started = time.perf_counter()
        classify_balanced_waste(sample)
        durations.append((time.perf_counter() - started) * 1000)
verification = {
    "tensorflow": tf.__version__,
    "gpu_devices": [device.name for device in tf.config.list_physical_devices("GPU")],
    "model_sha256": hashlib.sha256((RESULTS / "ecosort_weighted_12class.keras").read_bytes()).hexdigest(),
    "sample": str(SAMPLE), "decision": message_check3,
    "median_warm_prediction_ms": float(np.median(durations)),
    "scores": scores_check3,
    "scope": "Local inference smoke test only; not a new accuracy evaluation or full retraining.",
}
(PROJECT / "local_verification.json").write_text(json.dumps(verification, indent=2))
print(json.dumps(verification, indent=2))
display(sample.copy())


## 4. Open the local application

Start the original upload/webcam interface on http://127.0.0.1:7860. Only this Mac can access it; no Gradio public tunnel is created. Keep this notebook kernel running while using the application. Close any existing standalone EcoSort application before running this cell, because both use port 7860.


In [ ]:
if "demo3" in globals():
    demo3.close()
demo3 = gr.Interface(
    fn=classify_balanced_waste,
    inputs=gr.Image(sources=["upload", "webcam"], type="pil", label="Waste item"),
    outputs=[gr.Label(num_top_classes=12, label="Category scores"), gr.Textbox(label="Decision", interactive=False)],
    examples=[[str(SAMPLE)]],
    title="EcoSort AI", flagging_mode="never",
)
demo3.launch(share=False, server_name="127.0.0.1", server_port=7860, inline=False, inbrowser=False, prevent_thread_lock=True)
